In [1]:
import numpy as np
import pickle as pkl
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import silhouette_score

In [2]:
"""def get_data(dataset, method, e, b):
    if b == 1 and e == np.inf:
        filename = dataset + ".train.rating"
    elif b == 0:
        # FullDP Baseline
        filename = dataset + ".train_e" + str(e) + "_b" + str(b) + "_random_dp.rating"
    else:
        filename = dataset + ".train_e" + str(e) + "_b" + str(b) + "_" + method + ".rating"
    
    return pd.read_csv(dataset + "/" + filename, sep="\t", header=None, names=["user_id", "item_id", "rating"])"""

def get_data(dataset, method, e, b):
    if method.endswith("_dp"):
        if b == 1 and e == np.inf:
            filename = dataset + ".train.rating"
        elif b == 0:
            # FullDP Baseline
            filename = dataset + ".train_e" + str(e) + "_b" + str(b) + "_random_dp.rating"
        else:
            filename = dataset + ".train_e" + str(e) + "_b" + str(b) + "_" + method + ".rating"
    else:
        if b == 1:
            filename = dataset + ".train.rating"
        else:
            filename = dataset + ".train_b" + str(b) + "_" + method + ".rating"
    
    return pd.read_csv(dataset + "/" + filename, sep="\t", header=None, names=["user_id", "item_id", "rating"])
        

def get_stats(df):
    n_users = df["user_id"].nunique()
    n_items = df["item_id"].nunique()
    n_ratings = len(df)
    density = n_ratings / (n_users*n_items)
    n_ratings_per_user = df.groupby("user_id").size().median()
    n_ratings_per_item = df.groupby("item_id").size().median()
    
    
    print("No. Users %d" % n_users)
    print("No. Items %d" % n_items)
    print("No. Ratings %d" % n_ratings)
    print("Density %f" % density)
    print("Ratings per User %f" % n_ratings_per_user)
    print("Ratings per Item %f" % n_ratings_per_item)
    
    df.groupby("item_id").size().hist()
    
def compute_ister_score(dataset_df, user_info_df):
    merged_df = pd.merge(dataset_df, user_info_df, left_on="user_id:token", right_on="user_id:token")
    item_attr_interactions_df = merged_df[["item_id:token", "attr:token"]]
    item_attr_dist = item_attr_interactions_df.groupby(["item_id:token", "attr:token"]).size()
    attr_dist = item_attr_interactions_df.groupby("attr:token").size()
    igi_score = item_attr_dist / attr_dist
    
    # compute i_ster score
    ister_scores = dict()
    for iid in dataset_df["item_id:token"].unique():
        if 0 not in igi_score.loc[iid] or 1 not in igi_score.loc[iid]:
            ister_scores[iid] = 0
        else:
            diff = igi_score.loc[iid][0] - igi_score.loc[iid][1]
            ister_scores[iid] = diff / max(igi_score.loc[iid][0], igi_score.loc[iid][1])
    
    return ister_scores

In [3]:
DATASET = "ml1m"
PATH = DATASET + "/"
df = pd.read_csv(PATH + DATASET + ".train.rating", sep="\t", header=None)
df.columns = ["user_id:token", "item_id:token", "rating:float"]

user_attr_df = pd.read_csv(PATH + DATASET + ".userlist", sep="\t")
ister_scores = compute_ister_score(df, user_attr_df)

In [4]:
def measure_stereotypicality(u_ster, gender=0):
    if gender:
        return -u_ster
    else:
        return u_ster

In [5]:
data_df = get_data(dataset = "ml1m", method="random_del", e=0.1, b=1)
data_df["i_ster"] = data_df["item_id"].map(ister_scores)

data_df = pd.merge(data_df, user_attr_df, left_on="user_id", right_on="user_id:token")
data_df.drop(columns=["user_id:token"], inplace=True)
data_df.rename(columns={"attr:token": "attr"}, inplace=True)

data_df["u_ster"] = data_df["user_id"].map(data_df.groupby(["user_id"])["i_ster"].median())
print(data_df.groupby("attr")["u_ster"].mean())

attr
0    0.081728
1   -0.098018
Name: u_ster, dtype: float64


In [23]:
data_df = get_data(dataset = "ml1m", method="random_del", e=0.1, b=0.8)
data_df["i_ster"] = data_df["item_id"].map(ister_scores)

data_df = pd.merge(data_df, user_attr_df, left_on="user_id", right_on="user_id:token")
data_df.drop(columns=["user_id:token"], inplace=True)
data_df.rename(columns={"attr:token": "attr"}, inplace=True)

data_df["u_ster"] = data_df["user_id"].map(data_df.groupby(["user_id"])["i_ster"].median())
print(data_df.groupby("attr")["u_ster"].mean())

attr
0    0.080719
1   -0.097103
Name: u_ster, dtype: float64


In [24]:
data_df = get_data(dataset = "ml1m", method="random_del", e=0.1, b=0.6)
data_df["i_ster"] = data_df["item_id"].map(ister_scores)

data_df = pd.merge(data_df, user_attr_df, left_on="user_id", right_on="user_id:token")
data_df.drop(columns=["user_id:token"], inplace=True)
data_df.rename(columns={"attr:token": "attr"}, inplace=True)

data_df["u_ster"] = data_df["user_id"].map(data_df.groupby(["user_id"])["i_ster"].median())
print(data_df.groupby("attr")["u_ster"].mean())

attr
0    0.082374
1   -0.097541
Name: u_ster, dtype: float64


In [25]:
data_df = get_data(dataset = "ml1m", method="random_del", e=0.1, b=0.4)
data_df["i_ster"] = data_df["item_id"].map(ister_scores)

data_df = pd.merge(data_df, user_attr_df, left_on="user_id", right_on="user_id:token")
data_df.drop(columns=["user_id:token"], inplace=True)
data_df.rename(columns={"attr:token": "attr"}, inplace=True)

data_df["u_ster"] = data_df["user_id"].map(data_df.groupby(["user_id"])["i_ster"].median())
print(data_df.groupby("attr")["u_ster"].mean())

attr
0    0.081283
1   -0.096360
Name: u_ster, dtype: float64


In [26]:
data_df = get_data(dataset = "ml1m", method="random_del", e=0.1, b=0.2)
data_df["i_ster"] = data_df["item_id"].map(ister_scores)

data_df = pd.merge(data_df, user_attr_df, left_on="user_id", right_on="user_id:token")
data_df.drop(columns=["user_id:token"], inplace=True)
data_df.rename(columns={"attr:token": "attr"}, inplace=True)

data_df["u_ster"] = data_df["user_id"].map(data_df.groupby(["user_id"])["i_ster"].median())
print(data_df.groupby("attr")["u_ster"].mean())

attr
0    0.079518
1   -0.097687
Name: u_ster, dtype: float64


In [27]:
data_df = get_data(dataset = "ml1m", method="random_del", e=0.1, b=0)
data_df["i_ster"] = data_df["item_id"].map(ister_scores)

data_df = pd.merge(data_df, user_attr_df, left_on="user_id", right_on="user_id:token")
data_df.drop(columns=["user_id:token"], inplace=True)
data_df.rename(columns={"attr:token": "attr"}, inplace=True)

data_df["u_ster"] = data_df["user_id"].map(data_df.groupby(["user_id"])["i_ster"].median())
print(data_df.groupby("attr")["u_ster"].mean())

attr
0    0.074084
1   -0.075305
Name: u_ster, dtype: float64


In [28]:
data_df = get_data(dataset = "ml1m", method="random_del", e=0.1, b=0)
data_df["i_ster"] = data_df["item_id"].map(ister_scores)

data_df = pd.merge(data_df, user_attr_df, left_on="user_id", right_on="user_id:token")
data_df.drop(columns=["user_id:token"], inplace=True)
data_df.rename(columns={"attr:token": "attr"}, inplace=True)

#data_df["u_ster"] = data_df["user_id"].map(data_df.groupby(["user_id"])["i_ster"].median())
#print(data_df.groupby("attr")["u_ster"].mean())
data_df.groupby("user_id")["i_ster"].median()

user_id
0       0.190391
1       0.085640
2       0.149608
3      -0.178005
4      -0.171779
          ...   
6035    0.714464
6036   -0.140574
6037    0.613924
6038    0.692321
6039    0.456123
Name: i_ster, Length: 6040, dtype: float64

In [ ]:
data1_df = get_data(dataset = "ml1m", method="ister_dp", e=np.inf, b=1)
data1_df["i_ster"] = data1_df["item_id"].map(ister_scores)
data1_df = pd.merge(data1_df, user_attr_df, left_on="user_id", right_on="user_id:token")

data08_df = get_data(dataset = "ml1m", method="ister_dp", e=0.1, b=0.8)
data08_df["i_ster"] = data08_df["item_id"].map(ister_scores)
data08_df = pd.merge(data08_df, user_attr_df, left_on="user_id", right_on="user_id:token")

data06_df = get_data(dataset = "ml1m", method="ister_dp", e=0.1, b=0.6)
data06_df["i_ster"] = data06_df["item_id"].map(ister_scores)
data06_df = pd.merge(data06_df, user_attr_df, left_on="user_id", right_on="user_id:token")

data04_df = get_data(dataset = "ml1m", method="ister_dp", e=0.1, b=0.4)
data04_df["i_ster"] = data04_df["item_id"].map(ister_scores)
data04_df = pd.merge(data04_df, user_attr_df, left_on="user_id", right_on="user_id:token")

data02_df = get_data(dataset = "ml1m", method="ister_dp", e=0.1, b=0.2)
data02_df["i_ster"] = data02_df["item_id"].map(ister_scores)
data02_df = pd.merge(data02_df, user_attr_df, left_on="user_id", right_on="user_id:token")

data0_df = get_data(dataset = "ml1m", method="ister_dp", e=0.1, b=0)
data0_df["i_ster"] = data0_df["item_id"].map(ister_scores)
data0_df = pd.merge(data0_df, user_attr_df, left_on="user_id", right_on="user_id:token")

In [10]:
def analyze_changed_items(df1, df2):
    df = pd.merge(df1, df2, left_on="item_id", right_on="item_id")
    df["i_ster"] = df["item_id"].map(ister_scores)
    df["diff"] = df["count_y"] - df["count_x"]
    df.drop(columns=["count_x", "count_y"], inplace=True)

    added_items_df = df[df["diff"] > 0]
    deleted_items_df = df[df["diff"] < 0]

    print("Avg. I_ster Added: %f" % ((added_items_df["i_ster"] * added_items_df["diff"]).sum() / added_items_df["diff"].sum()))
    print("Avg. I_ster Deleted: %f" % ((deleted_items_df["i_ster"] * deleted_items_df["diff"]).sum() / deleted_items_df["diff"].sum()))


print("Beta 1 to 0.8, Females (I_ster > 0)")
df1 = data1_df[data1_df["attr:token"] == 0]["item_id"].value_counts().to_frame().reset_index()
df1.columns = ["item_id", "count"]
df2 = data08_df[data08_df["attr:token"] == 0]["item_id"].value_counts().to_frame().reset_index()
df2.columns = ["item_id", "count"]
analyze_changed_items(df1, df2)
print()
print("Beta 1 to 0.8, Males (I_ster < 0)")
df1 = data1_df[data1_df["attr:token"] == 1]["item_id"].value_counts().to_frame().reset_index()
df1.columns = ["item_id", "count"]
df2 = data08_df[data08_df["attr:token"] == 1]["item_id"].value_counts().to_frame().reset_index()
df2.columns = ["item_id", "count"]
analyze_changed_items(df1, df2)

print("==================================\n")
print("Beta 0.8 to 0.6, Females (I_ster > 0)")
df1 = data08_df[data08_df["attr:token"] == 0]["item_id"].value_counts().to_frame().reset_index()
df1.columns = ["item_id", "count"]
df2 = data06_df[data06_df["attr:token"] == 0]["item_id"].value_counts().to_frame().reset_index()
df2.columns = ["item_id", "count"]
analyze_changed_items(df1, df2)
print()
print("Beta 0.8 to 0.6, Males (I_ster < 0)")
df1 = data08_df[data08_df["attr:token"] == 1]["item_id"].value_counts().to_frame().reset_index()
df1.columns = ["item_id", "count"]
df2 = data06_df[data06_df["attr:token"] == 1]["item_id"].value_counts().to_frame().reset_index()
df2.columns = ["item_id", "count"]
analyze_changed_items(df1, df2)

print("==================================\n")
print("Beta 0.6 to 0.4, Females (I_ster > 0)")
df1 = data06_df[data06_df["attr:token"] == 0]["item_id"].value_counts().to_frame().reset_index()
df1.columns = ["item_id", "count"]
df2 = data04_df[data04_df["attr:token"] == 0]["item_id"].value_counts().to_frame().reset_index()
df2.columns = ["item_id", "count"]
analyze_changed_items(df1, df2)
print()
print("Beta 0.6 to 0.4, Males (I_ster < 0)")
df1 = data06_df[data06_df["attr:token"] == 1]["item_id"].value_counts().to_frame().reset_index()
df1.columns = ["item_id", "count"]
df2 = data04_df[data04_df["attr:token"] == 1]["item_id"].value_counts().to_frame().reset_index()
df2.columns = ["item_id", "count"]
analyze_changed_items(df1, df2)

print("==================================\n")
print("Beta 0.4 to 0.2, Females (I_ster > 0)")
df1 = data04_df[data04_df["attr:token"] == 0]["item_id"].value_counts().to_frame().reset_index()
df1.columns = ["item_id", "count"]
df2 = data02_df[data02_df["attr:token"] == 0]["item_id"].value_counts().to_frame().reset_index()
df2.columns = ["item_id", "count"]
analyze_changed_items(df1, df2)
print()
print("Beta 0.4 to 0.2, Males (I_ster < 0)")
df1 = data04_df[data04_df["attr:token"] == 1]["item_id"].value_counts().to_frame().reset_index()
df1.columns = ["item_id", "count"]
df2 = data02_df[data02_df["attr:token"] == 1]["item_id"].value_counts().to_frame().reset_index()
df2.columns = ["item_id", "count"]
analyze_changed_items(df1, df2)

print("==================================\n")
print("Beta 0.2 to 0, Females (I_ster > 0)")
df1 = data1_df[data1_df["attr:token"] == 0]["item_id"].value_counts().to_frame().reset_index()
df1.columns = ["item_id", "count"]
df2 = data08_df[data1_df["attr:token"] == 0]["item_id"].value_counts().to_frame().reset_index()
df2.columns = ["item_id", "count"]
analyze_changed_items(df1, df2)
print()
print("Beta 0.2 to 0, Males (I_ster < 0)")
df1 = data02_df[data02_df["attr:token"] == 1]["item_id"].value_counts().to_frame().reset_index()
df1.columns = ["item_id", "count"]
df2 = data0_df[data0_df["attr:token"] == 1]["item_id"].value_counts().to_frame().reset_index()
df2.columns = ["item_id", "count"]
analyze_changed_items(df1, df2)

print("==================================\n")

Beta 1 to 0.8, Females (I_ster > 0)
Avg. I_ster Added: -0.162440
Avg. I_ster Deleted: 0.547656

Beta 1 to 0.8, Males (I_ster < 0)
Avg. I_ster Added: 0.178394
Avg. I_ster Deleted: -0.518166

Beta 0.8 to 0.6, Females (I_ster > 0)
Avg. I_ster Added: -0.061374
Avg. I_ster Deleted: 0.284166

Beta 0.8 to 0.6, Males (I_ster < 0)
Avg. I_ster Added: 0.112592
Avg. I_ster Deleted: -0.289382

Beta 0.6 to 0.4, Females (I_ster > 0)
Avg. I_ster Added: 0.003849
Avg. I_ster Deleted: 0.104207

Beta 0.6 to 0.4, Males (I_ster < 0)
Avg. I_ster Added: 0.056759
Avg. I_ster Deleted: -0.122309

Beta 0.4 to 0.2, Females (I_ster > 0)
Avg. I_ster Added: 0.052622
Avg. I_ster Deleted: -0.055008

Beta 0.4 to 0.2, Males (I_ster < 0)
Avg. I_ster Added: -0.012542
Avg. I_ster Deleted: 0.068912

Beta 0.2 to 0, Females (I_ster > 0)
Avg. I_ster Added: -0.162440
Avg. I_ster Deleted: 0.547656

Beta 0.2 to 0, Males (I_ster < 0)
Avg. I_ster Added: -0.122694
Avg. I_ster Deleted: 0.324931



In [11]:
def get_cutoffs(df, beta, ascending=False):
    profile_sizes = df.groupby("user_id").size()
    cutoffs = []
    for uid, group_df in df.sort_values(by="i_ster", ascending=ascending).groupby("user_id"):
        i_sters = group_df["i_ster"].values
        cutoff_idx = int(profile_sizes.loc[uid] * (1-beta))
        cutoffs.append(i_sters[cutoff_idx-1])
    
    return cutoffs

df0 = data1_df[data1_df["attr:token"] == 0]
print("Females")
print("==================")
for beta in [0.8, 0.6, 0.4, 0.2, 0]:
    print("Cutoff for beta=%f: %f" % (beta, np.mean(get_cutoffs(df0, beta=beta))))
print()
df1 = data1_df[data1_df["attr:token"] == 1]
print("Males")
print("==================")
for beta in [0.8, 0.6, 0.4, 0.2, 0]:
    print("Cutoff for beta=%f: %f" % (beta, np.mean(get_cutoffs(df1, beta=beta, ascending=True))))

Females
Cutoff for beta=0.800000: 0.391612
Cutoff for beta=0.600000: 0.192250
Cutoff for beta=0.400000: 0.019342
Cutoff for beta=0.200000: -0.157538
Cutoff for beta=0.000000: -0.522991

Males
Cutoff for beta=0.800000: -0.357926
Cutoff for beta=0.600000: -0.188576
Cutoff for beta=0.400000: -0.031102
Cutoff for beta=0.200000: 0.156591
Cutoff for beta=0.000000: 0.564808


In [12]:
f_items = dict()
m_items = dict()
for iid, score in ister_scores.items():
    if score > 0:
        f_items[iid] = score
    elif score < 0:
        m_items[iid] = score

print(len(f_items), len(m_items))
print(np.mean(list(f_items.values())), np.mean(list(m_items.values())))

1679 1657
0.3765000527645328 -0.35940399881903523


In [13]:
np.mean(list(ister_scores.values()))

0.010044214690948974

In [14]:
(data1_df[data1_df["attr:token"] == 0]).groupby("user_id")["i_ster"].apply(lambda x: x > 0).mean()

C:\Users\pmuellner\AppData\Local\Temp\ipykernel_6288\1473313707.py:1: FutureWarning: Not prepending group keys to the result index of transform-like apply. In the future, the group keys will be included in the index, regardless of whether the applied function returns a like-indexed object.
To preserve the previous behavior, use

	>>> .groupby(..., group_keys=False)

To adopt the future behavior and silence this warning, use 

	>>> .groupby(..., group_keys=True)
  (data1_df[data1_df["attr:token"] == 0]).groupby("user_id")["i_ster"].apply(lambda x: x > 0).mean()


0.5750266269765409

In [15]:
(data1_df[data1_df["attr:token"] == 1]).groupby("user_id")["i_ster"].apply(lambda x: x < 0).mean()

C:\Users\pmuellner\AppData\Local\Temp\ipykernel_6288\1523730697.py:1: FutureWarning: Not prepending group keys to the result index of transform-like apply. In the future, the group keys will be included in the index, regardless of whether the applied function returns a like-indexed object.
To preserve the previous behavior, use

	>>> .groupby(..., group_keys=False)

To adopt the future behavior and silence this warning, use 

	>>> .groupby(..., group_keys=True)
  (data1_df[data1_df["attr:token"] == 1]).groupby("user_id")["i_ster"].apply(lambda x: x < 0).mean()


0.6092429579426031

In [8]:
DATASET = "bx"
PATH = DATASET + "/"
df = pd.read_csv(PATH + DATASET + ".train.rating", sep="\t", header=None)
df.columns = ["user_id:token", "item_id:token", "rating:float"]

user_attr_df = pd.read_csv(PATH + DATASET + ".userlist", sep="\t")

data1_df = get_data(dataset=DATASET, method="ister_dp", e=np.inf, b=1)
data1_df["i_ster"] = data1_df["item_id"].map(ister_scores)
data1_df = pd.merge(data1_df, user_attr_df, left_on="user_id", right_on="user_id:token")

print((data1_df[data1_df["attr:token"] == 0]).groupby("user_id")["i_ster"].apply(lambda x: x > 0).mean())
print((data1_df[data1_df["attr:token"] == 1]).groupby("user_id")["i_ster"].apply(lambda x: x < 0).mean())

ister_scores = compute_ister_score(df, user_attr_df)
f_items = dict()
m_items = dict()
for iid, score in ister_scores.items():
    if score > 0:
        f_items[iid] = score
    elif score < 0:
        m_items[iid] = score

print(len(f_items), len(m_items))
print(np.mean(list(f_items.values())), np.mean(list(m_items.values())))

C:\Users\pmuellner\AppData\Local\Temp\ipykernel_19092\2971609450.py:12: FutureWarning: Not prepending group keys to the result index of transform-like apply. In the future, the group keys will be included in the index, regardless of whether the applied function returns a like-indexed object.
To preserve the previous behavior, use

	>>> .groupby(..., group_keys=False)

To adopt the future behavior and silence this warning, use 

	>>> .groupby(..., group_keys=True)
  print((data1_df[data1_df["attr:token"] == 0]).groupby("user_id")["i_ster"].apply(lambda x: x > 0).mean())


0.40901955187669475


C:\Users\pmuellner\AppData\Local\Temp\ipykernel_19092\2971609450.py:13: FutureWarning: Not prepending group keys to the result index of transform-like apply. In the future, the group keys will be included in the index, regardless of whether the applied function returns a like-indexed object.
To preserve the previous behavior, use

	>>> .groupby(..., group_keys=False)

To adopt the future behavior and silence this warning, use 

	>>> .groupby(..., group_keys=True)
  print((data1_df[data1_df["attr:token"] == 1]).groupby("user_id")["i_ster"].apply(lambda x: x < 0).mean())


0.3399463075032989
2298 1881
0.45628948579432865 -0.4149196800130438
